*[cell 0]*

# LVL_1 Sweep Report — calibration on all data

Calibration of the LVL_1 knobs on the sweep written by `launch_LV1_sweep_random.py` (T2 and T3: every participant). The report reads the
sweep folder only (`SWEEP_DIR`, `[cell 1]`; its `_provenance.json` is checked by `sweep_provenance.py`). The one file
it may write there is `_script_lvl1.py`, the byte copy of the `lvl1.py` the sweep ran: only when the copy is missing
and `lvl1.py` still has the sweep's fingerprint.

**Conditions.** The manuscript calls the two conditions cohorts: T2 = Cohort A (CA), T3 = Cohort B (CB). The code,
file names, exports and the final report keep T2 and T3; only `make_replication_figures.ipynb` prints CA and CB.

Sections:

0. **Quality filters and effect-size definition**
1. **Which knobs matter** (Tables 1.1–1.4: SS shares of `smoothing_window` and `pelt_jump` per detector; the knob values at each detector's best configuration)
2. **Main effect of each knob**: the mean effect size at each knob level (figure)
3. **The winner**, by effect size, with both significance rates beside it
4. **Best solution per method**: each detector at its own optimal configuration
6. **Interactions**: `pelt_jump` x `smoothing_window` (heatmap)
7. **Summary**: the winner under each criterion, and the selection

**Outputs** (in `LVL_OUTS`): `<sweep>_report_{knob_ss, selection_freq, best_per_detector}.csv` and
`<sweep>_selection.json`, the detector and knob values the LVL1 notebook's `LVL_1_CONFIG` must hold (its calibration
check reads the file), `LVL_1_CONFIG_full` (every key of the script copy's `LVL_1_CONFIG`, with those knob values)
and `script_copy` (the path of the script copy). H1 is tested in the LVL1 notebook on its handoff; the calibration figure is drawn by
`make_replication_figures.ipynb`.

In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
SWEEP_DIR = "_sweep_lvl1_all"   # the sweep folder in SWEEPS_DIR (the launcher's --outdir), e.g. "_sweep_lvl1_all"

import ast, sys, json, re
import numpy as np
import pandas as pd
from pathlib import Path

# ---- locations: everything is inside /home/sjsuss626/EMST_files/sjs_dissertation ----------------
LVL_OUTS   = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/lvl_outs")   # this report's tables
SWEEPS_DIR = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/sweeps")     # the sweep folders
CODE_DIR   = Path("/home/sjsuss626/EMST_files/sjs_dissertation/code/practical_application_emst")        # lvl1.py

_dwb, sys.dont_write_bytecode = sys.dont_write_bytecode, True   # as python -B: no __pycache__/
from sweep_provenance import read_sweep, pipeline_fingerprint, grid_fingerprint
sys.dont_write_bytecode = _dwb

if SWEEP_DIR is None:
    raise SystemExit("SWEEP_DIR is not set (top of this cell): the --outdir of launch_LV1_sweep_random.py")
# Only a directory whose _provenance.json matches its result files is accepted.
SWEEP_DIR = SWEEPS_DIR / SWEEP_DIR
SWEEP_PROV = read_sweep(SWEEP_DIR, r"cfg\d{6}",
                        {k: k for k in ("pipeline_fingerprint", "lvl0_input", "condition", "subsample", "seed")})
CFG_FILES = sorted(SWEEP_DIR.glob("cfg[0-9][0-9][0-9][0-9][0-9][0-9].csv"))
if not CFG_FILES:
    raise SystemExit(f"no cfg*.csv in {SWEEP_DIR}")
# Completeness: config ids are grid positions 0 .. N-1, and the cfg_ columns of the result files (the swept knobs in
# the launcher's order, then its fixed knobs) rebuild the grid whose hash is _provenance.json "grid". The sweep is read
# only when its cfg files (each with its sessions_ file) are that whole grid. Otherwise the configurations that failed
# or never ran are listed, with the launcher's errors/ files for the folder, and the report stops.
ERR_DIR = CODE_DIR / "errors"


def _cfg_value(s):
    """A cfg_ value as lvl1.py parsed it (the result files hold repr() of each value)."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return s


_files = {int(f.stem[-6:]): f for f in CFG_FILES}
_n = max(_files) + 1
_gaps = [f"cfg{i:06d}" for i in range(_n) if i not in _files]
_incomplete = [f.stem for f in _files.values() if not f.with_name(f"sessions_{f.name}").is_file()]
_recs = {}
for _i, _f in _files.items():
    _r = pd.read_csv(_f, nrows=1, keep_default_na=False, dtype=str)
    _recs[_i] = [(c[4:], _cfg_value(_r[c].iloc[0])) for c in _r.columns if c.startswith("cfg_")]
_grid_ok = False
if not _gaps and not _incomplete:
    for _k in range(len(_recs[0]) + 1):          # the first _k cfg_ columns are the swept knobs
        _fixed = dict(_recs[0][_k:])
        if all(dict(_recs[i][_k:]) == _fixed for i in _recs) and grid_fingerprint(
                [[v for _, v in _recs[i][:_k]] for i in range(_n)] + [_fixed]) == SWEEP_PROV["grid"]:
            _grid_ok = True
            print(f"{SWEEP_DIR.name}: {_n} configurations = the launcher's whole grid (hash {SWEEP_PROV['grid']}; swept "
                  f"{[k for k, _ in _recs[0][:_k]]}), each with its sessions_ file")
            break
if not _grid_ok:
    _errs = sorted(f.name for f in ERR_DIR.glob(f"{SWEEP_DIR.name}_*.err"))
    raise SystemExit(
        f"{SWEEP_DIR.name} is not the launcher's whole grid: {len(_files)} cfg files with ids up to {_n - 1}, which do not "
        f"rebuild the grid hashed in _provenance.json ({SWEEP_PROV['grid']}). Missing configurations: "
        f"{', '.join(_gaps) or 'none below the highest id'}"
        f"{'; without a sessions_ file: ' + ', '.join(_incomplete) if _incomplete else ''}; errors/ files: "
        f"{', '.join(_errs) or 'none'}. Re-run the launcher on this folder (it resumes) before reading it.")
# the launcher's pooled table must hold exactly the rows of every configuration file
_pooled = SWEEP_DIR / "sweep_results.csv"
if not _pooled.is_file():
    raise SystemExit(f"{_pooled.name} is missing in {SWEEP_DIR}: the launcher writes it when the sweep finishes")
_n_cfg = sum(len(pd.read_csv(f)) for f in CFG_FILES)
if len(pd.read_csv(_pooled)) != _n_cfg:
    raise SystemExit(f"{_pooled.name} has a different number of rows than the {len(CFG_FILES)} configuration files "
                     f"({_n_cfg}): re-run the launcher on {SWEEP_DIR.name} to pool them")
TAB_STEM = LVL_OUTS / f"{SWEEP_DIR.name}_report"
REPORT_TABLES = {}


def export(key, frame):
    """Write one table to LVL_OUTS as <sweep>_report_<key>.csv and show it."""
    REPORT_TABLES[key] = frame
    frame.to_csv(f"{TAB_STEM}_{key}.csv", index=False)
    display(frame)


print(f"sweep {SWEEP_DIR.name}: {len(CFG_FILES)} configurations, {_n_cfg:,} rows")
print(f"provenance: lvl1.py {SWEEP_PROV['pipeline_fingerprint']}, {SWEEP_PROV['lvl0_input']}, "
      f"condition={SWEEP_PROV['condition']}")
if pipeline_fingerprint(CODE_DIR / "lvl1.py") != SWEEP_PROV["pipeline_fingerprint"]:
    print("WARNING: lvl1.py has changed since this sweep ran; the sweep describes the OLD pipeline.")
# The script the sweep ran: its byte copy SWEEP_DIR/_script_lvl1.py, which must have the sweep's fingerprint. A missing
# copy is written from lvl1.py, only when lvl1.py still has that fingerprint. Its LVL_1_CONFIG (the dict literal of
# NOTEBOOK CELL 9) is the base of the selection's LVL_1_CONFIG_full ([cell 23]).
SCRIPT_COPY = SWEEP_DIR / "_script_lvl1.py"
if not SCRIPT_COPY.is_file():
    if pipeline_fingerprint(CODE_DIR / "lvl1.py") != SWEEP_PROV["pipeline_fingerprint"]:
        raise SystemExit(f"{SCRIPT_COPY.name} is missing in {SWEEP_DIR} and lvl1.py does not have the sweep's "
                         f"fingerprint {SWEEP_PROV['pipeline_fingerprint']}: the script the sweep ran is not available")
    SCRIPT_COPY.write_bytes((CODE_DIR / "lvl1.py").read_bytes())
    SCRIPT_COPY.chmod(0o444)
    print(f"script copy written: {SCRIPT_COPY}")
if pipeline_fingerprint(SCRIPT_COPY) != SWEEP_PROV["pipeline_fingerprint"]:
    raise SystemExit(f"{SCRIPT_COPY} has fingerprint {pipeline_fingerprint(SCRIPT_COPY)}, not the sweep's "
                     f"{SWEEP_PROV['pipeline_fingerprint']}: it is not the script the sweep ran")
_cfg_node = [n for n in ast.parse(SCRIPT_COPY.read_bytes()).body if isinstance(n, ast.Assign)
             and [getattr(t, "id", None) for t in n.targets] == ["LVL_1_CONFIG"]]
if len(_cfg_node) != 1:
    raise SystemExit(f"{SCRIPT_COPY.name} has {len(_cfg_node)} top-level LVL_1_CONFIG assignments, not one")
SWEEP_LVL1_CONFIG = json.loads(json.dumps(ast.literal_eval(_cfg_node[0].value)))   # JSON values: tuples -> lists
print(f"script copy: {SCRIPT_COPY} (fingerprint {pipeline_fingerprint(SCRIPT_COPY)}; LVL_1_CONFIG: "
      f"{len(SWEEP_LVL1_CONFIG)} keys)")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))


In [ ]:
# [cell 2]
df_raw = pd.read_csv(SWEEP_DIR / "sweep_results.csv")   # pooled in cell 1 (provenance-checked)

df = df_raw

EFF_RAW = "avg_effsize"
EFF = EFF_RAW
SIG, CORR = "avg_sig%", "avg_corr_sig%"

# The effect-size column is test-neutral (*_effsize): validate_anova returns
# eta^2_H = (H - k + 1) / (n - k) under Kruskal-Wallis and omega-squared under ANOVA. Label from the
# test that was actually run so figures and printouts name the right statistic.
VALIDATE_TEST = (str(df["cfg_validate_test"].iloc[0]).strip("\"'").lower()
                 if "cfg_validate_test" in df.columns else "kruskal")
EFF_SYM = {"kruskal": "\u03b7\u00b2_H", "anova": "\u03c9\u00b2"}.get(VALIDATE_TEST, "effect size")
EFF_LONG = {"kruskal": "eta-squared H, (H - k + 1) / (n - k) (Kruskal-Wallis)",
            "anova": "omega-squared (one-way ANOVA)"}.get(VALIDATE_TEST, VALIDATE_TEST)
EFF_LABEL = f"avg {EFF_SYM}"
EFF_LABEL_FIG = EFF_LABEL.replace("\u03b7\u00b2_H", r"$\eta^2_H$")
KNOBS = [c for c in df.columns if c.startswith("cfg_")]
# knobs that actually vary in this sweep (the fixed overrides are constant)
VARY = [k for k in KNOBS if df[k].nunique() > 1]

print(f"rows: {len(df):,}   configs: {len(df)//max(df['method'].nunique(),1):,}   "
      f"methods: {df['method'].nunique()}")
print(f"effect-size column: {EFF}   reported as {EFF_LABEL} - {EFF_LONG}")
print(f"knobs that vary: {[k.replace('cfg_','') for k in VARY]}")
print(f"knobs held fixed: {[k.replace('cfg_','') for k in KNOBS if k not in VARY]}")

*[cell 3]*

## 0. Quality filters and effect-size definition

Three independent gates (set any to `None` to disable) plus a choice of which
effect size the whole report ranks on.

**Why the gates.** `blocks_cv` measures block-size imbalance, which is worth
screening on. But CV goes to 0 for a degenerate one-block-per-session solution,
so on its own it selects *against* segmentation: across this sweep
`corr(CV, n_validatable) = +0.69`. The two floors below stop that.

**The validation set.** Every method is validated on the same three features it
could segment on - `Adj RT Level (Stim)`, `Adj RT Var (win)`, `Entropy (win)` - so `avg_effsize` has a common
denominator across methods. The effect size of the segmentation features *is* the
test of whether segmentation worked. Coherence features and the cross-scale
`RT Variance` are not in the publication set.

**Why the effect-size choice.** Two of the three (`Adj RT Var (win)`, `Entropy (win)`) are
rolling statistics built with `derivative_filter_size`. Widening that window
autocorrelates them, which inflates Kruskal-Wallis H (it assumes independent
observations) with no change to block quality. Measured on `HMM_3_Adj_RT_Level_Stim`, whose
blocks barely move across the sweep: smoothed features rise 2.6-2.8x from `dfs=5` to
`dfs=19`; unsmoothed features move 1.02x. `EFF_MODE = "unsmoothed"` rebuilds the
effect size from `Adj RT Level (Stim)` alone, the one feature immune to that window.

When `derivative_filter_size` is held fixed (current sweep: 9) that inflation is
the same for every config, and `smoothing_window` only touches the detection signal,
so `EFF_MODE = "all"` is fair and keeps all three features.

`pelt_jump` only reaches PELT methods, so it is blanked (NaN) on MAD/HMM rows.


In [ ]:
# [cell 4]
# ---------------------------------------------------------------- filters
CV_MAX       = 0.60   # block-size CV ceiling (e.g., 0.30). None = off
MIN_BLOCKS   = None   # mean blocks/session floor. (e.g., 0.30) None = off
MIN_VAL_FRAC = None   # n_validatable / n_obs floor. None (0.90) = off

# ---------------------------------------------------------------- effect size
# "all"        -> avg_effsize as written (6 features; 3 are smoothed by dfs)
# "unsmoothed" -> average of the features that do NOT use derivative_filter_size
# "ac"         -> avg_effsize rescaled by an AR(1) effective-n factor (approximate)
EFF_MODE      = "all"
AC_WINDOW_COL = "cfg_derivative_filter_size"

SMOOTHED   = ["Entropy (win)_effsize", "Adj RT Var (win)_effsize"]   # built with derivative_filter_size
UNSMOOTHED = ["Adj RT Level (Stim)_effsize"]                    # immune to derivative_filter_size

df = df_raw.copy()

if EFF_MODE == "unsmoothed":
    cols = [c for c in UNSMOOTHED if c in df.columns]
    df["eff_unsmoothed"] = df[cols].mean(axis=1, skipna=True)
    EFF = "eff_unsmoothed"
    print(f"effect size = mean of {cols} (immune to derivative_filter_size)")
elif EFF_MODE == "ac":
    # rolling mean of width w -> adjacent values share w-1 inputs, rho ~ (w-1)/w,
    # AR(1) effective n = n*(1-rho)/(1+rho) ~ n/(2w-1). Rescales the statistic by
    # that factor. APPROXIMATE: H is not stored, so this scales eta^2_H directly.
    w = df[AC_WINDOW_COL].astype(float)
    df["eff_ac"] = df[EFF_RAW] / (2 * w - 1)
    EFF = "eff_ac"
    print(f"effect size = {EFF_RAW} / (2w-1), w = {AC_WINDOW_COL}  [approximate]")
else:
    EFF = EFF_RAW
    print(f"effect size = {EFF_RAW} (unadjusted; includes smoothed features)")

_before = len(df)
_gates = []
if CV_MAX is not None:
    df = df[df["blocks_cv"] <= CV_MAX];              _gates.append(f"blocks_cv<={CV_MAX}")
if MIN_BLOCKS is not None:
    df = df[df["blocks_mean"] >= MIN_BLOCKS];        _gates.append(f"blocks_mean>={MIN_BLOCKS}")
if MIN_VAL_FRAC is not None:
    df = df[df["n_validatable"] >= MIN_VAL_FRAC * df["n_obs"]]
    _gates.append(f"n_validatable>={MIN_VAL_FRAC}*n_obs")

print(f"filters: {' AND '.join(_gates) if _gates else 'none'}")
print(f"rows {_before:,} -> {len(df):,} ({100*len(df)/max(_before,1):.0f}% kept), "
      f"methods {df['method'].nunique()}, configs {len(df)//max(df['method'].nunique(),1):,}")
if df.empty:
    raise SystemExit("all rows filtered out - loosen CV_MAX / MIN_BLOCKS / MIN_VAL_FRAC")

# pelt_jump only reaches PELT_* methods; blank it elsewhere so MAD/HMM rows do not
# dilute its eta2 or report an arbitrary jump as their "best".
if "cfg_pelt_jump" in df.columns:
    df.loc[~df["method"].str.startswith("PELT"), "cfg_pelt_jump"] = np.nan

VARY = [k for k in KNOBS if df[k].nunique() > 1]
print(f"knobs that vary after filtering: {[k.replace('cfg_','') for k in VARY]}")


In [ ]:
# [cell 5]
import matplotlib.pyplot as plt
# --- plotting setup -------------------------------------------------------
# Categorical palette: fixed order, never cycled. Validated colorblind-safe
# (lightness band / chroma floor / CVD separation / normal-vision floor all pass).
# The contrast WARN is relieved by the direct labels + tables throughout.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
SEQ    = "#2a78d6"        # single hue for magnitude (never a rainbow)
INK, MUTED = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "legend.frameon": False, "font.size": 9,
})

def topn_methods(n=6):
    return (df.groupby("method")[EFF].mean()
              .sort_values(ascending=False).head(n).index.tolist())

*[cell 6]*

## 1. Which knobs matter at all?

Over each detector's configurations (all of the sweep), the share of the sum of squares of each criterion (η²_H,
Corr%) explained by `smoothing_window`, by `pelt_jump` and by their interaction (Tables 1.1–1.2), and how often each
knob value is a detector's best configuration (Tables 1.3–1.4). † marks the η²_H winner.

In [ ]:
# [cell 7]
# Tables 1.1-1.4. Over each detector's configurations (every configuration of the sweep): the SS share of
# smoothing_window, of pelt_jump and of their interaction (1 - the two shares) in each criterion, and the knob values at
# each detector's best configuration. pelt_jump only reaches the PELT detectors; the others have one active knob, so
# only the criterion's range is shown. Descriptive.
if not set(VARY) <= {"cfg_smoothing_window", "cfg_pelt_jump"}:
    raise SystemExit("Tables 1.1-1.4 decompose smoothing_window x pelt_jump; this sweep also varies "
                     f"{sorted(set(VARY) - {'cfg_smoothing_window', 'cfg_pelt_jump'})}")
sw1 = pd.concat([pd.read_csv(f).assign(cfg_file=f.name) for f in CFG_FILES], ignore_index=True)
sw1.loc[~sw1.method.str.startswith("PELT"), "cfg_pelt_jump"] = np.nan          # jump does not reach MAD / HMM
CRIT1 = {"η²_H": "avg_effsize", "Corr%": "avg_corr_sig%"}
cfg1 = (sw1.groupby(["method", "cfg_pelt_jump", "cfg_smoothing_window"], dropna=False)
           .agg(**{k: (v, "first") for k, v in CRIT1.items()}, cfg_file=("cfg_file", "first"),
                blocks_mean=("blocks_mean", "first"), n_validatable=("n_validatable", "first"), n_obs=("n_obs", "first"))
           .reset_index())
METHODS1 = sorted(cfg1.method.unique())
PELT1 = [m for m in METHODS1 if m.startswith("PELT")]
_best_eta = cfg1.sort_values("η²_H", ascending=False, kind="stable").groupby("method").first()
WINNER_LVL1 = _best_eta["η²_H"].idxmax()


def ss_shares(d, y, a="cfg_smoothing_window", b="cfg_pelt_jump"):
    """SS shares of two crossed knobs (one value per cell): a, b, and the remainder (a x b interaction)."""
    g = d[y].mean(); tot = ((d[y] - g) ** 2).sum()
    ssa = sum(len(x) * (x[y].mean() - g) ** 2 for _, x in d.groupby(a))
    ssb = sum(len(x) * (x[y].mean() - g) ** 2 for _, x in d.groupby(b))
    return tot, ssa, ssb


rows, pooled = [], {k: np.zeros(3) for k in CRIT1}
for m in METHODS1:
    d = cfg1[cfg1.method == m]
    for lab, col in CRIT1.items():
        r = {"Method": m + (" †" if m == WINNER_LVL1 else ""), "Criterion": lab, "Configs": len(d),
             "Min": d[lab].min(), "Max": d[lab].max(), "Range": d[lab].max() - d[lab].min()}
        if m in PELT1:
            tot, ssa, ssb = ss_shares(d, lab)
            r.update({"SS share smoothing": ssa / tot, "SS share pelt_jump": ssb / tot,
                      "SS share interaction": 1 - (ssa + ssb) / tot})
            pooled[lab] += [ssa, ssb, tot - ssa - ssb]
        else:
            r.update({"SS share smoothing": np.nan, "SS share pelt_jump": np.nan, "SS share interaction": np.nan})
        rows.append(r)
for lab in CRIT1:
    s = pooled[lab]
    rows.append({"Method": "All PELT detectors (within-detector SS pooled)", "Criterion": lab,
                 "Configs": int(cfg1.method.isin(PELT1).sum()), "Min": np.nan, "Max": np.nan, "Range": np.nan,
                 "SS share smoothing": s[0] / s.sum(), "SS share pelt_jump": s[1] / s.sum(),
                 "SS share interaction": s[2] / s.sum()})
export("knob_ss", pd.DataFrame(rows).sort_values(["Criterion", "Method"], kind="stable"))


def best_rows(col):
    out = {}
    for m in METHODS1:
        d = cfg1[cfg1.method == m]
        out[m] = d[np.isclose(d[col], d[col].max(), rtol=0, atol=1e-12)]
    return out


sel_rows = []
for lab, col in CRIT1.items():
    best = best_rows(lab)
    for knob, pool in (("smoothing_window", METHODS1), ("pelt_jump", PELT1)):
        levels = sorted(cfg1.loc[cfg1.method.isin(pool), f"cfg_{knob}"].dropna().unique())
        vals = {m: sorted(set(best[m][f"cfg_{knob}"].astype(float))) for m in pool}
        tied = [m for m in pool if len(vals[m]) > 1]
        r = {"Criterion": lab, "Knob": knob, "Detectors": len(pool)}
        for lv in levels:
            k = sum(1 for m in pool if vals[m] == [float(lv)])
            r[f"= {lv:g}"] = f"{k} ({100 * k / len(pool):.1f}%)"
        r["Tied"] = f"{len(tied)} ({100 * len(tied) / len(pool):.1f}%)"
        r["Tied detectors (values)"] = "; ".join(f"{m} ({', '.join(f'{v:g}' for v in vals[m])})" for m in tied) or "—"
        sel_rows.append(r)
export("selection_freq", pd.DataFrame(sel_rows))


*[cell 8]*

## 2. Main effect of each knob

Mean effect size at each knob level. Every series is both colored and directly
labeled, so identity never depends on color alone, and a method keeps the same
color in every panel.

`topn_methods(N)` sets which methods appear — except for `pelt_jump`, which only
reaches the PELT family and therefore shows **all four PELT methods** whatever
their rank.

Each point is a **marginal mean over the other knobs**: a `pelt_jump` point averages
the `smoothing_window` levels, and a `smoothing_window` point averages the
`pelt_jump` levels. For PELT that pulls the smoothing curve down, because jump=5 is
uniformly worst — read section 6's heatmap for the value at a specific pair.


In [ ]:
# [cell 9]
meths = topn_methods(5)          # controls every panel EXCEPT pelt_jump
PELT_METHS = sorted(m for m in df["method"].unique() if m.startswith("PELT"))

# pelt_jump only reaches the PELT family, so that panel shows ALL PELT methods
# regardless of rank (PELT_Adj_RT_Level_Stim is well outside the top N but its jump
# response is still the thing the knob is being judged on). Every other knob
# shows the top-N methods.
PANEL_METHS = {"cfg_pelt_jump": PELT_METHS}

# one colour per method across all panels, so a series keeps its colour
_all = list(dict.fromkeys(list(meths) + PELT_METHS))
COLOR = {m: SERIES[i % len(SERIES)] for i, m in enumerate(_all)}

n = len(VARY)
ncol = 2; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 3.5 * nrow), squeeze=False)

for ax, k in zip(axes.ravel(), VARY):
    for m in PANEL_METHS.get(k, meths):
        s = (df[df.method == m].groupby(k)[EFF].mean().sort_index())
        if s.empty:
            continue
        ax.plot(s.index.astype(str), s.values, marker="o", ms=4.5, lw=2,
                color=COLOR[m], label=m)
        ax.annotate(m, (len(s) - 1, s.values[-1]), textcoords="offset points",
                    xytext=(4, 0), fontsize=7, color=MUTED, va="center")
    ax.set_title(k.replace("cfg_", ""), fontweight="bold", loc="left")
    ax.set_xlabel(""); ax.set_ylabel(EFF)
    ax.legend(fontsize=7, ncol=2, loc="best")
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle("Main effect of each knob on effect size", fontweight="bold", x=0.01,
             ha="left", y=1.0)
plt.tight_layout(); plt.show()

print("Pooled across methods (mean effect size by level):")
for k in VARY:
    t = df.groupby(k)[[EFF, SIG, CORR]].mean().round(4)
    print(f"\n--- {k.replace('cfg_','')} ---"); print(t.to_string())

*[cell 10]*

## 3. The winner

Ranked by effect size, with **both** significance rates shown. See section 4
before trusting the top row.

In [ ]:
# [cell 11]
keep = [c for c in [EFF, SIG, CORR, "blocks_mean", "n_validatable", "n_obs"]
        if c in df.columns]

best_cfg = (df.groupby(["method"] + VARY, dropna=False)[keep].mean().reset_index()
              .sort_values(EFF, ascending=False).reset_index(drop=True))
print("TOP 20 method x config by effect size:")
display(best_cfg.head(20).round(4))

print("\nBest config for each method:")
display(best_cfg.sort_values(EFF, ascending=False)
                .groupby("method", as_index=False).first()
                .sort_values(EFF, ascending=False).round(4))

print("\nMethod ranking (averaged over all configs):")
display(df.groupby("method")[keep].mean().sort_values(EFF, ascending=False).round(4))

*[cell 12]*

## 4. Best solution per method

Each detector at the configuration maximising its effect size (precision) and at the configuration maximising
`avg_corr_sig%` (pervasiveness). A detector whose values barely move between the two tables has a configuration good
on both criteria.

Effect size is computed only on sessions with >= 2 blocks, so a method that often returns **one** block is scored on a
subset of sessions; `pct_single_block` = 1 - `n_validatable`/`n_obs` measures that. `Sig% - Corr%` is not an
independent tell: `Corr%` = `Sig%` x (`n_validatable`/`n_obs`). The flag below is a **diagnostic only**; it excludes
nothing from the winner.

`best_per_detector` (exported) is each detector's best configuration by η²_H over every configuration of the sweep;
the LVL1 notebook's H1 tests use these configurations for the T2+T3 calibration rows.

In [ ]:
# [cell 13]
SINGLE_BLOCK_MAX = 25   # % of sessions with <2 blocks above which the effect size is a subset estimate

FEAT = [c for c in df.columns if c.endswith("_effsize") and c != EFF_RAW]
FEAT_CORR = [c for c in df.columns if c.endswith("_corr_sig%") and c != CORR]

# one row per method x config, carrying the per-feature effect sizes as well
agg = (df.groupby(["method"] + VARY, dropna=False)[list(dict.fromkeys(keep + FEAT + FEAT_CORR))]
         .mean().reset_index())

# each method at ITS OWN best configuration
chk = (agg.sort_values(EFF, ascending=False, kind="stable").groupby("method", as_index=False).first()
          .set_index("method").sort_values(EFF, ascending=False))
chk["sig_minus_corr"] = chk[SIG] - chk[CORR]
chk["pct_single_block"] = 100 * (1 - chk["n_validatable"] / chk["n_obs"])
flag = chk["pct_single_block"] > SINGLE_BLOCK_MAX

# same table, optimized for pervasiveness instead of precision
chk_sig = (agg.sort_values(CORR, ascending=False, kind="stable").groupby("method", as_index=False).first()
              .set_index("method").sort_values(CORR, ascending=False))
chk_sig["pct_single_block"] = 100 * (1 - chk_sig["n_validatable"] / chk_sig["n_obs"])

_cols = [EFF, SIG, CORR, "blocks_mean", "pct_single_block"] + VARY
_star = lambda t: np.where(np.arange(len(t)) == 0, "★", "")
print(f"optimized for {EFF_LABEL}:")
display(chk[_cols].assign(**{"SUBSET?": np.where(flag, "<-- partial coverage", ""),
                             "BEST": _star(chk)}).round(3))
print(f"optimized for {CORR}:")
display(chk_sig[_cols].assign(**{"BEST": _star(chk_sig)}).round(3))

if flag.any():
    print(f"PARTIAL COVERAGE (pct_single_block > {SINGLE_BLOCK_MAX}%):",
          ", ".join(f"{m} ({chk.loc[m, 'pct_single_block']:.1f}%)" for m in chk.index[flag]))
else:
    print(f"Every method splits at least {100 - SINGLE_BLOCK_MAX}% of sessions.")

# Each detector's best configuration (highest η²_H over every configuration of the sweep): the LVL1 notebook's H1
# tests read the T2+T3 calibration sessions of these configurations. The quality gates (section 0) must not exclude any
# of them, or the gated ranking above and this table would describe different configurations.
BEST = _best_eta.reset_index()[["method", "cfg_file", "cfg_smoothing_window", "cfg_pelt_jump", "η²_H", "Corr%",
                                "blocks_mean", "n_validatable", "n_obs"]]
_gated = set(zip(df.method, df.cfg_smoothing_window.astype(float), df.cfg_pelt_jump.astype(float).fillna(-1)))
_out = [r.method for r in BEST.itertuples()
        if (r.method, float(r.cfg_smoothing_window), -1.0 if pd.isna(r.cfg_pelt_jump) else float(r.cfg_pelt_jump))
        not in _gated]
if _out:
    raise SystemExit(f"the quality gates (section 0) exclude the best configuration of {_out}; decide whether the "
                     "gates or the configurations should change")
export("best_per_detector", BEST)
# the gated per-configuration table and each detector's best configuration by each criterion, with its sweep file: the
# LVL1 calibration figure (make_replication_figures) and the LVL1 notebook's Corr% figure test read them
_cf = cfg1[["method", "cfg_smoothing_window", "cfg_pelt_jump", "cfg_file"]]
_with_file = lambda t: t.merge(_cf, on=["method", "cfg_smoothing_window", "cfg_pelt_jump"], how="left")
export("configs", _with_file(agg))
export("best_eta", _with_file(chk.reset_index()))
export("best_corr", _with_file(chk_sig.reset_index()))


*[cell 20]*

## 6. Interactions

- **`pelt_jump` x `smoothing_window`** - the current sweep. Mean over PELT methods only
  (pelt_jump is NaN elsewhere). The other panels are skipped when their knobs are fixed.

In [ ]:
# [cell 21]
def heat(a, b, title):
    ca, cb = f"cfg_{a}", f"cfg_{b}"
    if ca not in df.columns or cb not in df.columns:
        print(f"skip {a} x {b}: not in results"); return
    if df[ca].nunique() < 2 or df[cb].nunique() < 2:
        print(f"skip {a} x {b}: not varied"); return
    piv = df.pivot_table(index=ca, columns=cb, values=EFF, aggfunc="mean")
    fig, ax = plt.subplots(figsize=(1.0 * len(piv.columns) + 3.4,
                                    0.55 * len(piv.index) + 2.6))
    im = ax.imshow(piv.values, cmap="Blues", aspect="auto")   # sequential, one hue
    ax.set_xticks(range(len(piv.columns)), piv.columns)
    ax.set_yticks(range(len(piv.index)), piv.index)
    ax.set_xlabel(b); ax.set_ylabel(a)
    ax.set_title(title, fontweight="bold", loc="left")
    ax.grid(False)
    for i in range(piv.shape[0]):                              # values on cells
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > np.nanmean(piv.values) else INK)
    fig.colorbar(im, ax=ax, label=EFF, shrink=0.85)
    plt.tight_layout(); plt.show()
    display(piv.round(4))

heat("pelt_jump", "smoothing_window", "pelt_jump x smoothing_window  (PELT methods)")
heat("min_block_size", "pelt_jump", "min_block_size x pelt_jump")
heat("derivative_filter_size", "mad_window_size",
     "derivative_filter_size x mad_window_size")

*[cell 22]*

## 7. Summary

The winner under each optimisation target, with its configuration, and the selection written for the LVL1 notebook.

In [ ]:
# [cell 23]
def report_winner(tbl, criterion, label):
    win = tbl.index[0]
    row = tbl.loc[win]
    print("=" * 72)
    print(f"WINNER by {label}: {win}")
    print("=" * 72)
    print(f"  {EFF_LABEL:16} {row[EFF]:.4f}")
    print(f"  {SIG:16} {row[SIG]:.1f}%")
    print(f"  {CORR:16} {row[CORR]:.1f}%")
    print(f"  blocks_mean      {row['blocks_mean']:.2f}")
    print(f"  pct_single_block {row['pct_single_block']:.1f}%")
    print()
    print("  best configuration:")
    knobs = {}
    for k in VARY:
        if k == "cfg_pelt_jump" and pd.isna(row[k]):
            # PELT rows at the winner's value of every OTHER varied knob (smoothing_window,
            # rt_threshold_na_handling, ...), so the jump is not averaged across them.
            _p = df[df["method"].str.startswith("PELT")]
            for _o in VARY:
                if _o != k and pd.notna(row[_o]):
                    _p = _p[_p[_o] == row[_o]]
            _j = _p.groupby(k)[criterion].mean().sort_values(ascending=False)
            _ru = _p.groupby("method")[criterion].mean().sort_values(ascending=False)
            print(f"    {k.replace('cfg_',''):28} = {_j.index[0]:g}   "
                  f"# runner-up @ the winner's other knobs ({_ru.index[0]}, {_ru.iloc[0]:.4f}); "
                  f"jumps {', '.join(f'{j:g}:{v:.4f}' for j, v in _j.items())}")
            knobs[k.replace("cfg_", "")] = _j.index[0]
        else:
            print(f"    {k.replace('cfg_',''):28} = {row[k]}")
            knobs[k.replace("cfg_", "")] = row[k]
    return win, knobs


w_eff, KNOBS_EFF = report_winner(chk, EFF, EFF_LABEL)
print()
w_sig, _ = report_winner(chk_sig, CORR, CORR)
print()
print("=" * 72)
if w_eff == w_sig:
    print(f"Both criteria agree on {w_eff}.")
else:
    print(f"Criteria disagree: {EFF_LABEL} -> {w_eff}, {CORR} -> {w_sig}.")
    print(f"  {w_eff:24} {EFF_LABEL} {chk.loc[w_eff, EFF]:.4f}   {CORR} {chk.loc[w_eff, CORR]:.1f}%")
    print(f"  {w_sig:24} {EFF_LABEL} {chk_sig.loc[w_sig, EFF]:.4f}   {CORR} {chk_sig.loc[w_sig, CORR]:.1f}%")
if w_eff != WINNER_LVL1:
    raise SystemExit(f"the gated winner {w_eff} is not the η²_H winner over every configuration ({WINNER_LVL1})")

# The selection the LVL1 notebook must use: the η²_H winner and the knob values of its best configuration. The LVL1
# notebook's calibration check compares LVL_1_CONFIG with this file. LVL_1_CONFIG_full = every key of the sweep
# script's LVL_1_CONFIG (the script copy, [cell 1]) with these knob values; script_copy = the copy's path.
_py = lambda v: int(v) if isinstance(v, (float, np.floating)) and float(v).is_integer() else (
    float(v) if isinstance(v, (float, np.floating)) else v)
SELECTION = {"sweep": SWEEP_DIR.name, "pipeline_fingerprint": SWEEP_PROV["pipeline_fingerprint"],
             "lvl0_input": SWEEP_PROV["lvl0_input"], "condition": SWEEP_PROV["condition"], "criterion": EFF,
             "detector": w_eff, "config": _best_eta.loc[w_eff, "cfg_file"].removesuffix(".csv"),
             "LVL_1_CONFIG": {k: _py(v) for k, v in KNOBS_EFF.items()}}
SELECTION["script_copy"] = str(SCRIPT_COPY)
SELECTION["LVL_1_CONFIG_full"] = {**SWEEP_LVL1_CONFIG, **SELECTION["LVL_1_CONFIG"]}
with open(LVL_OUTS / f"{SWEEP_DIR.name}_selection.json", "w") as _f:
    json.dump(SELECTION, _f, indent=2)
print(f"\nselection -> {LVL_OUTS / (SWEEP_DIR.name + '_selection.json')}")
print(json.dumps(SELECTION, indent=1))
